# Adding the number of synsets a verb has to verb+case data

In this notebook we add the number of synsets a verb has to the database. We do this to see whether the certainty of a verb's dependent in said case being one semantic type can be correlated to the amount of meanings that word has. So basically is it more likely that **a verb has different valency patterns because it has multiple meanings** OR **because one meaning can have multiple valency patterns**

This is done by:
1. extracting verbs from a database table and putting them into a dataframe
2. finding Wordnet synset counts for each verb
3. classifying the verbs into 4 classes based on the counts: not in Estonian Wordnet, 1 meaning, 2-3 meanings, >3 meanings
4. adding the count classifications to the verb dataframe
5. turning the dataframe into a database table
6. importing the synset_count column from the newly created table into a previously existing table used for creating hoverplots

In [1]:
from estnltk.wordnet import Wordnet
import sqlite3
import pandas as pd

## Functions

In [2]:
#extract verbs from database statistics table and put them in a dataframe 
def verb_to_df(table_name):
    
    # database file path
    filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"
    
    #Connect to database
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()

    #query verbs from database table
    query = f"SELECT DISTINCT verb, verb_compound FROM {table_name}"

    #put results into dataframe
    df_verbs = pd.read_sql(query, conn)

    #combine verb and compound, strip extra spaces
    df_verbs['full_verb'] = (df_verbs['verb'] + ' ' + df_verbs['verb_compound']).str.strip()

    #close database connection
    conn.close()

    return df_verbs

In [3]:
#find Wordnet synset counts for each verb, classify into 4 classes, add to dataframe
def synset_count_df(dataframe):
    
    #get verbs out as a list
    fullverb = dataframe['full_verb'].tolist()

    #create wordnet object
    wn = Wordnet()
    
    #find synset count for each verb, put counts into 3 classes
    synset_count = []
    for verb in fullverb:
        count = len(wn[verb, 'v'])
        if count == 0:
            synset_count.append('not in Estonian Wordnet')
        elif count == 1:
            synset_count.append('1')
        elif count == 2 or count == 3:
            synset_count.append('2-3')
        else:
            synset_count.append('>3')

    #add synset count to dataframe
    dataframe['synset_count'] = synset_count

    return dataframe

In [4]:
#add synset counts to database statistics table
def synset_count_db(table_name, dataframe):
    
    #Connect to database
    filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"
    conn = sqlite3.connect(filename)
    cursor = conn.cursor()

    #insert all values into the temp table
    dataframe.to_sql('temp_synset', conn, if_exists="replace", index=False)

    #add synset count column to statistics table
    cursor.execute(f"""
        ALTER TABLE {table_name}
        ADD COLUMN synset_count TEXT;
    """)

    #fill synset count column with data from temporary table
    cursor.execute(
        f"""
        UPDATE {table_name}
        SET synset_count = (
            SELECT synset_count
            FROM temp_synset
            WHERE 
                temp_synset.verb = {table_name}.verb
                AND temp_synset.verb_compound = {table_name}.verb_compound
    );
    """)

    cursor.execute("DROP TABLE IF EXISTS temp_synset")

    # Commit and close
    conn.commit()
    conn.close()

In [5]:
#go through all steps to add synset counts to database table
def synsets_to_db(table_name):

    #extract verbs from database table and put them in a dataframe 
    df_verbs = verb_to_df(table_name)

    #find Wordnet synset counts for each verb, classify into 4 classes, add them to dataframe for each verb
    df_synsets = synset_count_df(df_verbs)

    #add a verbs's synset counts to database table
    synset_count_db(table_name, df_synsets)

## Add synset counts to database table by running functions
User has to specify what table they want to add synset counts to

In [ ]:
synsets_to_db('verb_case_log_location')

In [34]:
synsets_to_db('verb_case_log_location_time')

In [6]:
synsets_to_db('verb_adv_log_location')